###Prepare data
download data and make them gzipped:
gzip /dbfs/mnt/bd/openuniverselake/aisraw/aisdata/csv/aisdk-2022*.csv

read the gzipped data, rename columns and change data types

In [0]:
import os

csv_dir = '/mnt/bd/openuniverselake/aisraw/aisdata/csv'
delta_dir = csv_dir.replace('/csv', '/delta')
csv_view = "CREATE OR REPLACE TEMPORARY VIEW ais_csv USING csv OPTIONS ( path '**CSV**',   header 'true' );"
ais_view = """CREATE OR REPLACE TEMPORARY VIEW ais
                AS
                SELECT DISTINCT to_timestamp(`# Timestamp`, 'dd/MM/yyyy HH:mm:ss') as rec_time
                    , `Type of mobile` AS mobile_type
                    , cast( MMSI as INTEGER) AS mmsi
                    , cast(Latitude as NUMERIC(9,6)) AS latitude , CAST( Longitude as NUMERIC(9,6)) AS longitude
                    , `Navigational status` AS nav_status
                    , CAST( ROT as double) AS rot
                    , CAST( SOG as double) AS sog
                    , CAST( COG as double) AS cog
                    , CAST( Heading as INTEGER) AS heading
                    , IMO imo, Callsign callsign, Name name, `Ship type` ship_type, `Cargo type` cargo_type
                    , CAST( Width as double) AS ship_width
                    , CAST( Length as double) AS ship_length
                    , `Type of position fixing device` device
                    , CAST( Draught as double) AS draught
                    , Destination destination,  ETA eta, `Data source type` source_type
                    , CAST( A as INTEGER) AS a
                    , CAST( B as INTEGER) AS b
                    , CAST( C as INTEGER) AS c
                    , CAST( D as INTEGER) AS d 
                    , year(to_timestamp(`# Timestamp`, 'dd/MM/yyyy HH:mm:ss')) as yyyy
                    , date_format(to_timestamp(`# Timestamp`, 'dd/MM/yyyy HH:mm:ss'), 'yyyy-MM') as yyyy_mm
                    , date_format(to_timestamp(`# Timestamp`, 'dd/MM/yyyy HH:mm:ss'), 'yyyy-MM-dd') as yyyy_mm_dd
                FROM ais_csv; """

csv_files = [os.path.join(csv_dir, f) for f in os.listdir(f"/dbfs/{csv_dir}") if os.path.isfile(os.path.join(f"/dbfs/{csv_dir}", f))]

print(f"Target directory {delta_dir}")
for f in csv_files:
    replace_where = f"`yyyy_mm_dd` == '{f.replace(csv_dir, '').replace('/', '').replace('.csv', '').replace('.gz', '').replace('aisdk-', '')}'"  # https://stackoverflow.com/questions/70261756/databricks-overwriting-entire-table-instead-of-adding-new-partition
    print(f"Convert {f} ({replace_where})...")
    sql_view = csv_view.replace('**CSV**', f)
    spark.sql(sql_view)
    spark.sql(ais_view)
    df = spark.sql("select * from ais")
    df.coalesce(1).write.partitionBy("yyyy", "yyyy_mm", "yyyy_mm_dd").option("maxRecordsPerFile", 20000000).format("delta").mode("overwrite").option("replaceWhere", replace_where).save(delta_dir)

Post fix, if DISTINCT above forgotten (11min)
INSERT OVERWRITE TABLE ${schema_name}.messages
PARTITION (yyyy, yyyy_mm, yyyy_mm_dd)
SELECT DISTINCT * FROM ${schema_name}.messages;

In [0]:
%sql
SELECT * FROM delta.`/mnt/bd/openuniverselake/aisraw/aisdata/delta` LIMIT 10;